# Portable execution configuration

This public copy was prepared from the executed JPCO revision notebook.

Machine-specific absolute paths and saved execution outputs were removed.
The original numerical summaries are provided separately in this revision
package.

By default, the notebook treats its current working directory as the
revision root. If required, set the following environment variables before
starting Jupyter:

- `GW_GLITCH_REVISION_ROOT`: directory containing the revision scripts and
  result subdirectories;
- `GW_GLITCH_PROJECT_ROOT`: parent `GW_Glitch` project directory containing
  historical processed datasets;
- `GW_GLITCH_RESEARCH_ROOT`: parent research directory;
- `GW_GLITCH_IMAGE_PROJECT_ROOT`: archived Gravity Spy image-project root.

The full source datasets, trained checkpoints, prediction arrays, and local
execution logs are not stored in Git.


In [ ]:
import os
from pathlib import Path

PORTABLE_REVISION_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_REVISION_ROOT",
        Path.cwd(),
    )
).expanduser().resolve()

PORTABLE_PROJECT_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_PROJECT_ROOT",
        PORTABLE_REVISION_ROOT.parent.parent,
    )
).expanduser().resolve()

PORTABLE_RESEARCH_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_RESEARCH_ROOT",
        PORTABLE_PROJECT_ROOT.parent,
    )
).expanduser().resolve()

PORTABLE_DESKTOP_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_DESKTOP_ROOT",
        PORTABLE_RESEARCH_ROOT.parent,
    )
).expanduser().resolve()

PORTABLE_IMAGE_PROJECT_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_IMAGE_PROJECT_ROOT",
        PORTABLE_RESEARCH_ROOT / "Glitch_Image",
    )
).expanduser().resolve()

print("Portable revision root:", PORTABLE_REVISION_ROOT)
print("Portable project root:", PORTABLE_PROJECT_ROOT)
print("Portable research root:", PORTABLE_RESEARCH_ROOT)
print("Portable image-project root:", PORTABLE_IMAGE_PROJECT_ROOT)


In [ ]:
# Cell 1 — Check environment and dataset package
# This cell does not train a model or evaluate test data.

from pathlib import Path
import importlib
import json
import platform
import sys

PROJECT_DIR = (
    Path.home()
    / "desktop"
    / "research"
    / "gw_glitch"
    / "revision_2026"
)

DATA_DIR = PROJECT_DIR / "jpco_o3_splits_v1"

print("Python:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Operating system:", platform.platform())
print("Architecture:", platform.machine())
print("Notebook working directory:", Path.cwd())
print("Dataset directory:", DATA_DIR)

print("\n--- Package checks ---")

package_errors = {}

for name in ["numpy", "pandas", "sklearn", "xgboost"]:
    try:
        module = importlib.import_module(name)
        print(f"{name}: {module.__version__}")
    except Exception as error:
        package_errors[name] = str(error)
        print(f"{name}: IMPORT FAILED — {error}")

print("\n--- Required files ---")

required_files = [
    "manifest.json",
    "o3_primary.csv.gz",
    "split_assignments.csv.gz",
    "tuning_random_split.csv.gz",
    "split_summary.csv",
]

missing_files = []

for filename in required_files:
    path = DATA_DIR / filename
    if path.is_file():
        print(f"FOUND: {filename}")
    else:
        missing_files.append(filename)
        print(f"MISSING: {filename}")

if not missing_files:
    with (DATA_DIR / "manifest.json").open() as file:
        manifest = json.load(file)

    print("\nPrimary dataset rows:", f"{manifest['primary_rows']:,}")
    print("Original nine features:")
    print(manifest["feature_sets"]["original_nine"])

if package_errors or missing_files:
    print("\nCHECK NEEDS ATTENTION: share the output below this cell.")
else:
    print("\nREADY: environment imports and required files verified.")

In [ ]:
# Cell 2 — Run the pilot using env1 in a fresh Python process
# A separate process makes its peak-memory measurement easier to interpret.

from pathlib import Path
import subprocess
import sys

SCRIPT = PROJECT_DIR / "xgboost_cpu_pilot.py"
OUTPUT_ROOT = PROJECT_DIR / "pilot_outputs"

assert SCRIPT.is_file(), f"Save the downloaded script here: {SCRIPT}"

command = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--data-dir", str(DATA_DIR),
    "--output-root", str(OUTPUT_ROOT),
]

process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            "The pilot stopped. Share the error output above."
        )
except KeyboardInterrupt:
    process.terminate()
    process.wait()
    print("\nPilot stopped.")

In [ ]:
# Cell 3 — Full-training-partition resource pilot

from pathlib import Path
import subprocess
import sys

PROJECT_DIR = (
    Path.home()
    / "Desktop"
    / "Research"
    / "GW_Glitch"
    / "revision_2026"
)

DATA_DIR = PROJECT_DIR / "jpco_o3_splits_v1"
SCRIPT = PROJECT_DIR / "xgboost_cpu_full_pilot.py"
OUTPUT_ROOT = PROJECT_DIR / "pilot_outputs"

assert SCRIPT.is_file(), f"Save the downloaded script here: {SCRIPT}"
assert DATA_DIR.is_dir(), f"Dataset folder not found: {DATA_DIR}"

command = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--data-dir", str(DATA_DIR),
    "--output-root", str(OUTPUT_ROOT),
]

process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)

    if process.wait() != 0:
        raise RuntimeError(
            "The pilot stopped. Share the error output above."
        )

except KeyboardInterrupt:
    process.terminate()
    process.wait()
    print("\nPilot stopped.")

In [ ]:
# Cell 4 — Four-condition XGBoost search and validation

from pathlib import Path
import subprocess
import sys

PROJECT_DIR = (
    Path.home()
    / "Desktop"
    / "Research"
    / "GW_Glitch"
    / "revision_2026"
)

SCRIPT = PROJECT_DIR / "xgboost_revision_stage1.py"
DATA_DIR = PROJECT_DIR / "jpco_o3_splits_v1"
OUTPUT_DIR = PROJECT_DIR / "xgboost_stage1_results"

assert SCRIPT.is_file(), f"Save the script here: {SCRIPT}"

command = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
]

process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)

    if process.wait() != 0:
        raise RuntimeError(
            "The run stopped. Share the error output; "
            "completed folds are saved."
        )

except KeyboardInterrupt:
    process.terminate()
    process.wait()
    print("\nStopped. Rerun this cell to resume completed-fold checkpoints.")

In [ ]:
# Cell 5 — Frozen three-seed test evaluation

from pathlib import Path
import subprocess
import sys

PROJECT_DIR = (
    Path.home()
    / "Desktop"
    / "Research"
    / "GW_Glitch"
    / "revision_2026"
)

SCRIPT = PROJECT_DIR / "xgboost_revision_stage2.py"
assert SCRIPT.is_file(), f"Save the script here: {SCRIPT}"

command = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--data-dir", str(PROJECT_DIR / "jpco_o3_splits_v1"),
    "--stage1-dir", str(PROJECT_DIR / "xgboost_stage1_results"),
    "--output-dir", str(PROJECT_DIR / "xgboost_stage2_results"),
]

process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)

    if process.wait() != 0:
        raise RuntimeError("The run stopped. Share the error output.")

except KeyboardInterrupt:
    process.terminate()
    process.wait()
    print("\nStopped. Completed-seed checkpoints are preserved.")

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(str(PORTABLE_REVISION_ROOT))
script = ROOT / "mlp_revision_cpu_pilot.py"
assert script.is_file(), f"Save the downloaded script here: {script}"

command = [
    sys.executable, "-u", str(script),
    "--data-dir", str(ROOT / "jpco_o3_splits_v1"),
    "--output-root", str(ROOT / "pilot_outputs"),
]

with subprocess.Popen(
    command,
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError("Pilot stopped. Share the error printed above.")

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(str(PORTABLE_REVISION_ROOT))

for filename in ["mlp_revision_stage1.py", "xgboost_revision_stage1.py"]:
    assert (ROOT / filename).is_file(), f"Missing: {ROOT / filename}"

command = [
    sys.executable, "-u", str(ROOT / "mlp_revision_stage1.py"),
    "--data-dir", str(ROOT / "jpco_o3_splits_v1"),
    "--output-dir", str(ROOT / "mlp_stage1_results"),
]

with subprocess.Popen(
    command,
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError("Search stopped. Share the error printed above.")

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(str(PORTABLE_REVISION_ROOT))

for filename in [
    "mlp_revision_stage2.py",
    "mlp_revision_stage1.py",
    "xgboost_revision_stage1.py",
]:
    assert (ROOT / filename).is_file(), f"Missing: {ROOT / filename}"

command = [
    sys.executable, "-u", str(ROOT / "mlp_revision_stage2.py"),
    "--data-dir", str(ROOT / "jpco_o3_splits_v1"),
    "--stage1-dir", str(ROOT / "mlp_stage1_results"),
    "--output-dir", str(ROOT / "mlp_stage2_results"),
]

with subprocess.Popen(
    command,
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError("Evaluation stopped. Share the error printed above.")

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(str(PORTABLE_REVISION_ROOT))
script = ROOT / "fttransformer_environment_check.py"
assert script.is_file(), f"Missing: {script}"

command = [
    sys.executable, "-u", str(script),
    "--output-dir", str(ROOT / "environment_checks"),
]

with subprocess.Popen(
    command,
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError("Check stopped. Share the error above.")

## FT-Transformer — CPU resource pilot

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(str(PORTABLE_REVISION_ROOT))
SCRIPT = ROOT / "fttransformer_cpu_pilot.py"
assert SCRIPT.is_file(), f"Missing script: {SCRIPT}"

command = [
    sys.executable, "-u", str(SCRIPT),
    "--data-dir", str(ROOT / "jpco_o3_splits_v1"),
    "--output-root", str(ROOT / "pilot_outputs"),
]

with subprocess.Popen(
    command,
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()

if return_code:
    raise RuntimeError(
        f"Pilot exited with code {return_code}. Share the error output."
    )

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(str(PORTABLE_REVISION_ROOT))
SCRIPT = ROOT / "fttransformer_revision_stage1.py"
OUTPUT = ROOT / "fttransformer_stage1"
LOG = ROOT / "fttransformer_stage1_console.log"

assert SCRIPT.is_file(), f"Missing: {SCRIPT}"
assert (ROOT / "xgboost_revision_stage1.py").is_file()

command = [
    sys.executable, "-u", str(SCRIPT),
    "--data-dir", str(ROOT / "jpco_o3_splits_v1"),
    "--output-dir", str(OUTPUT),
]

process = subprocess.Popen(
    command,
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Prevent idle sleep while the Python process runs.
awake = subprocess.Popen(
    ["/usr/bin/caffeinate", "-i", "-w", str(process.pid)]
)

try:
    with LOG.open("a", buffering=1) as log:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
    return_code = process.wait()
finally:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    awake.terminate()
    awake.wait()

if return_code:
    raise RuntimeError(f"Run stopped. Share the final error from {LOG}")

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(str(PORTABLE_REVISION_ROOT))
SCRIPT = ROOT / "fttransformer_revision_stage2.py"
OUTPUT = ROOT / "fttransformer_stage2"
LOG = ROOT / "fttransformer_stage2_console.log"

for filename in [
    "fttransformer_revision_stage2.py",
    "fttransformer_revision_stage1.py",
    "xgboost_revision_stage1.py",
]:
    assert (ROOT / filename).is_file(), f"Missing: {filename}"

command = [
    sys.executable, "-u", str(SCRIPT),
    "--data-dir", str(ROOT / "jpco_o3_splits_v1"),
    "--stage1-dir", str(ROOT / "fttransformer_stage1"),
    "--output-dir", str(OUTPUT),
]

process = subprocess.Popen(
    command,
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

awake = subprocess.Popen(
    ["/usr/bin/caffeinate", "-i", "-w", str(process.pid)]
)

try:
    with LOG.open("a", buffering=1) as log:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
    return_code = process.wait()
finally:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    awake.terminate()
    awake.wait()

if return_code:
    raise RuntimeError(f"Run stopped. Share the final error from {LOG}")

## Common permutation importance — Referee 2, Comment 1

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(str(PORTABLE_REVISION_ROOT))
SCRIPT = ROOT / "common_permutation_importance.py"
LOG = ROOT / "permutation_importance_console.log"

assert SCRIPT.is_file(), f"Missing: {SCRIPT}"

process = subprocess.Popen(
    [sys.executable, "-u", str(SCRIPT), "--root", str(ROOT)],
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

awake = subprocess.Popen(
    ["/usr/bin/caffeinate", "-i", "-w", str(process.pid)]
)

try:
    with LOG.open("a", buffering=1) as log:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
    return_code = process.wait()
finally:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    awake.terminate()
    awake.wait()

if return_code:
    raise RuntimeError(f"Run stopped. Share the final error from {LOG}")

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.executable)

for package in ("torch", "xgboost", "shap", "captum", "pytorch-tabular"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)
SCRIPT = ROOT / "native_attribution_pilot.py"

assert SCRIPT.is_file(), f"Missing script: {SCRIPT}"
assert (ROOT / "common_permutation_importance.py").is_file()
assert (ROOT / "permutation_importance_v1" / "protocol.json").is_file()

log_dir = ROOT / "run_logs"
log_dir.mkdir(exist_ok=True)
log_path = log_dir / (
    "native_attribution_pilot_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".log"
)

command = [
    sys.executable, "-u", str(SCRIPT),
    "--root", str(ROOT),
]

print("Python:", sys.executable)
print("Log:", log_path)
print("Starting attribution pilot...", flush=True)

with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    awake = subprocess.Popen(
        ["/usr/bin/caffeinate", "-i", "-w", str(process.pid)]
    )

    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        awake.terminate()
        awake.wait()

if return_code:
    raise RuntimeError(
        f"Pilot stopped with exit code {return_code}. "
        f"Share the final error lines. Log: {log_path}"
    )

print("\nFinished. The output directory is printed above.")

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)
SCRIPT = ROOT / "native_attribution_full_v2.py"

required_files = [
    SCRIPT,
    ROOT / "native_attribution_full.py",
    ROOT / "native_attribution_pilot.py",
    ROOT / "common_permutation_importance.py",
    ROOT / "permutation_importance_v1" / "protocol.json",
    ROOT / "native_attribution_v1" / "protocol.json",
]

for path in required_files:
    assert path.is_file(), f"Missing required file: {path}"

log_dir = ROOT / "run_logs"
log_dir.mkdir(exist_ok=True)

log_path = log_dir / (
    "native_attribution_full_v2_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".log"
)

command = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--root",
    str(ROOT),
]

print("Python:", sys.executable)
print("Script:", SCRIPT)
print("Log:", log_path)
print("Existing valid v1 checkpoints will be reused.")
print("Starting adaptive attribution recovery...", flush=True)

with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    awake = subprocess.Popen(
        ["/usr/bin/caffeinate", "-i", "-w", str(process.pid)]
    )

    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        return_code = process.wait()

    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise

    finally:
        awake.terminate()
        awake.wait()

if return_code:
    raise RuntimeError(
        f"Recovery stopped with exit code {return_code}. "
        f"Share failure.txt and the final log lines. Log: {log_path}"
    )

print("\nAdaptive native-attribution analysis finished.")
print("Results:", ROOT / "native_attribution_v2")

## Operational Complexity Benchmark — Referee 2, Comment 2

This analysis evaluates serialized model size, reconstruction-bundle size,
process memory, CPU inference latency, throughput, and architecture-specific
structural metrics for the frozen XGBoost, MLP, and FT-Transformer models.

No models are trained, tuned, or selected in this analysis.

In [ ]:
from pathlib import Path
import hashlib
import sys

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
).resolve()

SCRIPT = ROOT / "complexity_benchmark_v1.py"

required_files = [
    SCRIPT,
    ROOT / "common_permutation_importance.py",
    ROOT / "native_attribution_pilot.py",
    ROOT / "xgboost_revision_stage1.py",
    ROOT / "xgboost_revision_stage2.py",
    ROOT / "mlp_revision_stage1.py",
    ROOT / "mlp_revision_stage2.py",
    ROOT / "fttransformer_revision_stage1.py",
    ROOT / "fttransformer_revision_stage2.py",
    ROOT / "jpco_o3_splits_v1" / "manifest.json",
    ROOT / "jpco_o3_splits_v1" / "o3_primary.csv.gz",
    ROOT / "permutation_importance_v1" / "protocol.json",
    ROOT / "permutation_importance_v1" / "cohorts" / "random_split.csv.gz",
]

missing_files = [
    str(path) for path in required_files
    if not path.is_file()
]

assert not missing_files, (
    "Missing required files:\n" + "\n".join(missing_files)
)

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

EXPECTED_SCRIPT_SHA256 = (
    "4b3c64b95c82682e8cca206be5fc06c9544b783b58e71dbbdc5f37b1447a7092"
)

actual_script_sha256 = sha256(SCRIPT)

print("Python executable:", sys.executable)
print("Project root:", ROOT)
print("Benchmark script:", SCRIPT)
print("Script SHA-256:", actual_script_sha256)

assert actual_script_sha256 == EXPECTED_SCRIPT_SHA256, (
    "The benchmark script differs from the reviewed version.\n"
    f"Expected: {EXPECTED_SCRIPT_SHA256}\n"
    f"Found:    {actual_script_sha256}"
)

print("\nPREFLIGHT PASSED")
print("No training or tuning will be performed.")

In [ ]:
from datetime import datetime, timezone
import subprocess

LOG_DIR = ROOT / "run_logs"
LOG_DIR.mkdir(exist_ok=True)

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

LOG_PATH = LOG_DIR / (
    f"complexity_benchmark_{timestamp}.log"
)

command = [
    "/usr/bin/caffeinate",
    "-dimsu",
    sys.executable,
    str(SCRIPT),
    "--root",
    str(ROOT),
]

LOG_HANDLE = LOG_PATH.open("w")

PROCESS = subprocess.Popen(
    command,
    cwd=ROOT,
    stdout=LOG_HANDLE,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

print("Benchmark started.")
print("Process ID:", PROCESS.pid)
print("Log file:", LOG_PATH)
print(
    "Output directory:",
    ROOT / "complexity_benchmark_v1",
)
print("\nKeep the Mac plugged in with the lid open.")
print("Do not run the launch cell again while this process is active.")

In [ ]:
status = PROCESS.poll()

if status is None:
    print("STATUS: RUNNING")
else:
    print(f"STATUS: FINISHED — exit code {status}")

if LOG_PATH.is_file():
    log_text = LOG_PATH.read_text(errors="replace")
    print("\n--- Latest log output ---\n")
    print(log_text[-8000:])

In [ ]:
assert len(complexity_per_seed) == 9
assert set(complexity_per_seed["model"]) == {
    "XGBoost",
    "MLP",
    "FT-Transformer",
}
assert set(complexity_per_seed["seed"]) == {42, 43, 44}
assert complexity_per_seed["prediction_replay_passed"].all()

# Three model-level complexity summaries
assert len(complexity_summary) == 3

# 3 models × 3 seeds × 4 batch sizes
assert len(latency_per_seed) == 36
assert not latency_per_seed.duplicated(
    ["model", "seed", "batch_size"]
).any()

assert set(latency_per_seed["model"]) == {
    "XGBoost",
    "MLP",
    "FT-Transformer",
}
assert set(latency_per_seed["seed"]) == {42, 43, 44}
assert latency_per_seed["batch_size"].nunique() == 4

# 3 models × 4 batch sizes, aggregated over seeds
assert len(latency_summary) == 12
assert not latency_summary.duplicated(
    ["model", "batch_size"]
).any()

assert set(latency_summary["model"]) == {
    "XGBoost",
    "MLP",
    "FT-Transformer",
}
assert latency_summary["batch_size"].nunique() == 4
assert latency_summary["training_seeds"].eq(3).all()

print("SUMMARY")
print(json.dumps(summary, indent=2))

print("\nCOMPLEXITY SUMMARY")
display(complexity_summary)

print("\nLATENCY PER SEED")
display(latency_per_seed)

print("\nLATENCY SUMMARY")
display(latency_summary)

print("\nCOMPLETION CHECK PASSED")

## Replacement Figure 6 — operational efficiency

In [ ]:
from pathlib import Path
import hashlib

ROOT = Path(str(PORTABLE_REVISION_ROOT))
SCRIPT = ROOT / "make_operational_complexity_figure.py"
SOURCE = ROOT / "complexity_benchmark_v1"

required = [
    SCRIPT,
    SOURCE / "summary.json",
    SOURCE / "complexity_summary.csv",
    SOURCE / "latency_summary.csv",
]

for path in required:
    assert path.exists(), f"MISSING: {path}"
    print(f"FOUND: {path}")

script_sha256 = hashlib.sha256(SCRIPT.read_bytes()).hexdigest()
print("Script SHA-256:", script_sha256)

assert script_sha256 == (
    "3ce7b70945f96cffc8e3782df283bbce"
    "3c3db1f6405b0cc62445fb0fe91cd495"
)

print("PREFLIGHT PASSED")

In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    str(SCRIPT),
    "--root",
    str(ROOT),
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

assert result.returncode == 0, (
    f"Figure build failed with exit code {result.returncode}"
)

In [ ]:
import json
from IPython.display import Image, display

OUT = ROOT / "complexity_figure_v1"

expected = [
    OUT / "fig6_operational_efficiency.pdf",
    OUT / "fig6_operational_efficiency.png",
    OUT / "table_operational_complexity.tex",
    OUT / "fig6_operational_efficiency_source.csv",
    OUT / "figure_build_summary.json",
]

for path in expected:
    assert path.exists(), f"MISSING: {path}"
    assert path.stat().st_size > 0, f"EMPTY: {path}"
    print(f"READY: {path.name} ({path.stat().st_size:,} bytes)")

build = json.loads(
    (OUT / "figure_build_summary.json").read_text()
)

assert build["completed"] is True
assert build["training_or_inference_performed"] is False
assert build["models"] == [
    "XGBoost",
    "MLP",
    "FT-Transformer",
]
assert build["training_seeds"] == [42, 43, 44]

display(
    Image(
        filename=str(
            OUT / "fig6_operational_efficiency.png"
        ),
        width=1100,
    )
)

print("COMPLETION CHECK PASSED")